Setup and Imports

In [1]:
!pip install tensorflow-datasets -q

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io
import os
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

Data Acquisition

In [3]:
BASE_URL = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/"

URLS = {
    "confirmed": BASE_URL + "time_series_covid19_confirmed_global.csv",
    "deaths":    BASE_URL + "time_series_covid19_deaths_global.csv",
    "recovered": BASE_URL + "time_series_covid19_recovered_global.csv"
}

def load_raw(url):
  response = requests.get(url)
  response.raise_for_status()
  return pd.read_csv(io.StringIO(response.text))

raw = {key: load_raw(url) for key, url in URLS.items()}
print("Successful Data Load:")
for k, df in raw.items():
  print(f"{k}: {df.shape[0]} rows x {df.shape[1]} cols")

Successful Data Load:
confirmed: 289 rows x 1147 cols
deaths: 289 rows x 1147 cols
recovered: 274 rows x 1147 cols


Schema Exploration

In [4]:
for name, df in raw.items():
  print(f"\n{'='*50}")
  print(f"Dataset: {name.upper()}")
  # print(f"Shape: {df.shape[0]} rows x {df.shape[1]} cols")
  print(f"Shape: {df.shape}")
  print(f"\nFirst 10 cols (Dates) {list(df.columns[10:])}")
  print(f"\nLast 10 cols (Dates) {list(df.columns[-10:])}")
  print(f"\nNull counts (non-date cols):")
  print(df[['Province/State', 'Country/Region', 'Lat', 'Long']].isnull().sum())


Dataset: CONFIRMED
Shape: (289, 1147)

First 10 cols (Dates) ['1/28/20', '1/29/20', '1/30/20', '1/31/20', '2/1/20', '2/2/20', '2/3/20', '2/4/20', '2/5/20', '2/6/20', '2/7/20', '2/8/20', '2/9/20', '2/10/20', '2/11/20', '2/12/20', '2/13/20', '2/14/20', '2/15/20', '2/16/20', '2/17/20', '2/18/20', '2/19/20', '2/20/20', '2/21/20', '2/22/20', '2/23/20', '2/24/20', '2/25/20', '2/26/20', '2/27/20', '2/28/20', '2/29/20', '3/1/20', '3/2/20', '3/3/20', '3/4/20', '3/5/20', '3/6/20', '3/7/20', '3/8/20', '3/9/20', '3/10/20', '3/11/20', '3/12/20', '3/13/20', '3/14/20', '3/15/20', '3/16/20', '3/17/20', '3/18/20', '3/19/20', '3/20/20', '3/21/20', '3/22/20', '3/23/20', '3/24/20', '3/25/20', '3/26/20', '3/27/20', '3/28/20', '3/29/20', '3/30/20', '3/31/20', '4/1/20', '4/2/20', '4/3/20', '4/4/20', '4/5/20', '4/6/20', '4/7/20', '4/8/20', '4/9/20', '4/10/20', '4/11/20', '4/12/20', '4/13/20', '4/14/20', '4/15/20', '4/16/20', '4/17/20', '4/18/20', '4/19/20', '4/20/20', '4/21/20', '4/22/20', '4/23/20', '4/24/2

In [5]:
meta_cols = ['Province/State', 'Country/Region', 'Lat', 'Long']
date_cols = [c for c in raw['confirmed'].columns if c not in meta_cols]

print(f"Date range   : {date_cols[0]}  →  {date_cols[-1]}")
print(f"Total days   : {len(date_cols)}")
print(f"Countries    : {raw['confirmed']['Country/Region'].nunique()}")
print(f"Provinces    : {raw['confirmed']['Province/State'].nunique()}")
print(f"\nRow counts per dataset:")
for name, df in raw.items():
    print(f"  {name:12s}: {len(df)} rows")
print(f"\nData format  : WIDE (one column per date — will reshape to LONG in cleaning)")

Date range   : 1/22/20  →  3/9/23
Total days   : 1143
Countries    : 201
Provinces    : 91

Row counts per dataset:
  confirmed   : 289 rows
  deaths      : 289 rows
  recovered   : 274 rows

Data format  : WIDE (one column per date — will reshape to LONG in cleaning)


## Data Coverage Summary

| Metric | Data |
|---|---|
| Date Range | 1/22/20 – 3/9/23 |
| Total Days | 1,143 |
| Unique Countries | 201 |
| Unique Provinces/States | 91 |
| Confirmed Rows | 289 |
| Deaths Rows | 289 |
| Recovered Rows | 274 |
| Format | Wide (to be reshaped to Long) |

**Next step:** Clean and reshape to long format, aggregate to country level,
and engineer daily new case/death features.

Data Cleaning

In [6]:
# Column definitions (reusable)
META_COLS = ['Province/State', 'Country/Region', 'Lat', 'Long']
date_cols = [c for c in raw['confirmed'].columns if c not in META_COLS]

print(f"Meta columns : {META_COLS}")
print(f"Date columns : {len(date_cols)} total ({date_cols[0]} → {date_cols[-1]})")

Meta columns : ['Province/State', 'Country/Region', 'Lat', 'Long']
Date columns : 1143 total (1/22/20 → 3/9/23)


In [7]:
# Reshape DF from wide to long
def melt_df(df, value_name):
    return (
        df.drop(columns=['Lat', 'Long'])                          # drop coords
          .fillna({'Province/State': 'Unknown'})                  # fill null provinces
          .melt(
              id_vars=['Province/State', 'Country/Region'],
              var_name='date',
              value_name=value_name
          )
    )

long = {
    'confirmed' : melt_df(raw['confirmed'], 'confirmed'),
    'deaths'    : melt_df(raw['deaths'],    'deaths'),
    'recovered' : melt_df(raw['recovered'], 'recovered'),
}

for name, df in long.items():
    print(f"{name:12s}: {df.shape}  columns: {list(df.columns)}")

confirmed   : (330327, 4)  columns: ['Province/State', 'Country/Region', 'date', 'confirmed']
deaths      : (330327, 4)  columns: ['Province/State', 'Country/Region', 'date', 'deaths']
recovered   : (313182, 4)  columns: ['Province/State', 'Country/Region', 'date', 'recovered']


In [8]:
# Parse dates and aggregate to country level
def aggregate_to_country(df, value_col):
    df['date'] = pd.to_datetime(df['date'], format='%m/%d/%y')
    return (
        df.groupby(['Country/Region', 'date'], as_index=False)[value_col]
          .sum()
    )

country = {
    'confirmed' : aggregate_to_country(long['confirmed'], 'confirmed'),
    'deaths'    : aggregate_to_country(long['deaths'],    'deaths'),
    'recovered' : aggregate_to_country(long['recovered'], 'recovered'),
}

for name, df in country.items():
    print(f"{name:12s}: {df.shape}")
    print(f"  date dtype : {df['date'].dtype}")
    print(f"  sample     :\n{df[df['Country/Region'] == 'US'].head(3).to_string()}\n")

confirmed   : (229743, 3)
  date dtype : datetime64[ns]
  sample     :
       Country/Region       date  confirmed
212598             US 2020-01-22          1
212599             US 2020-01-23          1
212600             US 2020-01-24          2

deaths      : (229743, 3)
  date dtype : datetime64[ns]
  sample     :
       Country/Region       date  deaths
212598             US 2020-01-22       0
212599             US 2020-01-23       0
212600             US 2020-01-24       0

recovered   : (229743, 3)
  date dtype : datetime64[ns]
  sample     :
       Country/Region       date  recovered
212598             US 2020-01-22          0
212599             US 2020-01-23          0
212600             US 2020-01-24          0



In [9]:
# Merge confirmed, death, recover df's to one master df
from functools import reduce

dfs = [country['confirmed'], country['deaths'], country['recovered']]
master = reduce(lambda left, right: pd.merge(left, right, on=['Country/Region', 'date'], how='left'), dfs)

# Rename for clarity
master.columns = ['country', 'date', 'confirmed', 'deaths', 'recovered']

# Fill recovered nulls with 0 (countries missing from recovered dataset)
master['recovered'] = master['recovered'].fillna(0).astype(int)

print(f"Master shape : {master.shape}")
print(f"Columns      : {list(master.columns)}")
print(f"Nulls        :\n{master.isnull().sum()}")
print(f"\nSample:\n{master[master['country'] == 'US'].head(5).to_string()}")

Master shape : (229743, 5)
Columns      : ['country', 'date', 'confirmed', 'deaths', 'recovered']
Nulls        :
country      0
date         0
confirmed    0
deaths       0
recovered    0
dtype: int64

Sample:
       country       date  confirmed  deaths  recovered
212598      US 2020-01-22          1       0          0
212599      US 2020-01-23          1       0          0
212600      US 2020-01-24          2       0          0
212601      US 2020-01-25          2       0          0
212602      US 2020-01-26          5       0          0


In [10]:
# Validate monotonic counts (cumulative should never decrease)
def find_violations(df, col):
    violations = []
    for country_name, group in df.groupby('country'):
        group = group.sort_values('date')
        diffs = group[col].diff()
        bad = diffs[diffs < 0]
        if len(bad) > 0:
            violations.append({
                'country': country_name,
                'violations': len(bad),
                'worst_drop': diffs.min()
            })
    return pd.DataFrame(violations).sort_values('worst_drop')

print("Checking confirmed cases...")
conf_violations = find_violations(master, 'confirmed')
print(f"  Countries with drops: {len(conf_violations)}")

print("\nChecking deaths...")
death_violations = find_violations(master, 'deaths')
print(f"  Countries with drops: {len(death_violations)}")

if len(conf_violations) > 0:
    print(f"\nWorst confirmed offenders:\n{conf_violations.head(5).to_string()}")
if len(death_violations) > 0:
    print(f"\nWorst death offenders:\n{death_violations.head(5).to_string()}")

Checking confirmed cases...
  Countries with drops: 84

Checking deaths...
  Countries with drops: 80

Worst confirmed offenders:
       country  violations  worst_drop
21      France          12  -348840.00
70       Spain           5   -74347.00
60        Peru          10   -65286.00
33  Kazakhstan           2   -59375.00
2    Australia           3   -26856.00

Worst death offenders:
       country  violations  worst_drop
70       Spain           8    -1918.00
59        Peru           4     -679.00
35  Kyrgyzstan           1     -443.00
12      Canada           3     -358.00
14       China           1     -310.00


## Monotonic Validation Findings

84 countries show drops in confirmed cases, 80 in deaths.
These are NOT data errors, they are official retrospective corrections
by national health authorities (e.g. France removed ~348k duplicate cases,
Spain revised death counts multiple times).

In [11]:
# Feature engineering
master = master.sort_values(['country', 'date']).reset_index(drop=True)

# Daily new cases and deaths (diff w/in each country, clip negatives to 0)
master['new_cases'] = (
    master.groupby('country')['confirmed']
          .diff()
          .clip(lower=0)
          .fillna(0)
          .astype(int)
)

master['new_deaths'] = (
    master.groupby('country')['deaths']
          .diff()
          .clip(lower=0)
          .fillna(0)
          .astype(int)
)

# 7-day rolling average of new cases per country
master['rolling_7day'] = (
    master.groupby('country')['new_cases']
          .transform(lambda x: x.rolling(7, min_periods=1).mean())
          .round(2)
)

print(f"Columns now  : {list(master.columns)}")
print(f"Shape        : {master.shape}")
print(f"\nUS sample (first wave):")
print(master[master['country'] == 'US'][['date','confirmed','new_cases','new_deaths','rolling_7day']].iloc[60:65].to_string())

Columns now  : ['country', 'date', 'confirmed', 'deaths', 'recovered', 'new_cases', 'new_deaths', 'rolling_7day']
Shape        : (229743, 8)

US sample (first wave):
             date  confirmed  new_cases  new_deaths  rolling_7day
212658 2020-03-22      34944       8919         128       4533.14
212659 2020-03-23      46096      11152         187       5916.71
212660 2020-03-24      56714      10618         243       7171.71
212661 2020-03-25      68841      12127         333       8524.57
212662 2020-03-26      86662      17821         417      10428.43


In [12]:
# Final validation and dtype cleanup

print("Null check:")
print(master.isnull().sum())

# Confirm no negative new_cases or new_deaths
print(f"\nNegative new_cases  : {(master['new_cases'] < 0).sum()}")
print(f"Negative new_deaths : {(master['new_deaths'] < 0).sum()}")

# Confirm dtypes
print(f"\nDtypes:\n{master.dtypes}")

# Summary stats preview
print(f"\nOverall totals (final day):")
final_day = master[master['date'] == master['date'].max()]
print(f"  Global confirmed : {final_day['confirmed'].sum():,}")
print(f"  Global deaths    : {final_day['deaths'].sum():,}")
print(f"  Countries        : {final_day['country'].nunique()}")

Null check:
country         0
date            0
confirmed       0
deaths          0
recovered       0
new_cases       0
new_deaths      0
rolling_7day    0
dtype: int64

Negative new_cases  : 0
Negative new_deaths : 0

Dtypes:
country                 object
date            datetime64[ns]
confirmed                int64
deaths                   int64
recovered                int64
new_cases                int64
new_deaths               int64
rolling_7day           float64
dtype: object

Overall totals (final day):
  Global confirmed : 676,570,149
  Global deaths    : 6,881,802
  Countries        : 201


## Cleaning Pipeline Complete

Final dataset: 229,743 rows × 8 columns
- Zero nulls across all columns
- Zero negative daily values (clipped at source)
- Cumulative totals verified against published figures:
  - Global confirmed: 676,570,149
  - Global deaths: 6,881,802
  - Countries: 201
- Date range: 2020-01-22 → 2023-03-09

In [13]:
master.to_csv('covid19_master_cleaned.csv', index=False)
print("Saved: covid19_master_cleaned.csv")
print(f"Shape: {master.shape}")

Saved: covid19_master_cleaned.csv
Shape: (229743, 8)


TDFS Construction

In [15]:
# Custom TDFS construction
import tensorflow_datasets as tfds
import tensorflow as tf
import pandas as pd
import numpy as np

class Covid19Dataset(tfds.core.GeneratorBasedBuilder):
    """TensorFlow Dataset for JHU CSSE COVID-19 global time-series data."""

    VERSION = tfds.core.Version('1.0.0')
    DESCRIPTION = """
        COVID-19 global time-series dataset built from Johns Hopkins University
        CSSE data. Contains daily confirmed cases, deaths, recovered, new cases,
        new deaths, and 7-day rolling averages per country from 2020-01-22 to
        2023-03-09. Cleaned and aggregated to country level.
    """

    def _info(self):
        return tfds.core.DatasetInfo(
            builder=self,
            description=self.DESCRIPTION,
            features=tfds.features.FeaturesDict({
                'country'     : tfds.features.Text(),
                'date'        : tfds.features.Text(),
                'confirmed'   : tf.int64,
                'deaths'      : tf.int64,
                'recovered'   : tf.int64,
                'new_cases'   : tf.int64,
                'new_deaths'  : tf.int64,
                'rolling_7day': tf.float32,
            }),
            supervised_keys=None,
        )

    def _split_generators(self, dl_manager):
        # Use the master DataFrame already in memory
        # Sort by date and do a chronological 80/20 split
        df = master.sort_values('date').reset_index(drop=True)
        split_idx = int(len(df) * 0.8)

        train_df = df.iloc[:split_idx]
        eval_df  = df.iloc[split_idx:]

        print(f"Total rows   : {len(df)}")
        print(f"Train rows   : {len(train_df)} ({train_df['date'].min().date()} → {train_df['date'].max().date()})")
        print(f"Eval rows    : {len(eval_df)}  ({eval_df['date'].min().date()} → {eval_df['date'].max().date()})")

        return {
            'train': self._generate_examples(train_df),
            'eval' : self._generate_examples(eval_df),
        }

    def _generate_examples(self, df):
        for idx, row in df.iterrows():
            yield idx, {
                'country'     : str(row['country']),
                'date'        : str(row['date'].date()),
                'confirmed'   : int(row['confirmed']),
                'deaths'      : int(row['deaths']),
                'recovered'   : int(row['recovered']),
                'new_cases'   : int(row['new_cases']),
                'new_deaths'  : int(row['new_deaths']),
                'rolling_7day': float(row['rolling_7day']),
            }

print("Covid19Dataset class defined successfully.")

Covid19Dataset class defined successfully.


In [16]:
# Build dataset
import os
import tempfile

# Use a temp directory in Colab
data_dir = '/content/tfds_covid19'
os.makedirs(data_dir, exist_ok=True)

# Instantiate the builder
builder = Covid19Dataset(data_dir=data_dir)

# Build info (no download needed — data is already in memory)
builder.download_and_prepare(
    download_config=tfds.download.DownloadConfig(
        manual_dir=data_dir
    )
)

print("\nDataset built successfully.")
print(builder.info)

Total rows   : 229743
Train rows   : 183794 (2020-01-22 → 2022-07-24)
Eval rows    : 45949  (2022-07-24 → 2023-03-09)


Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /content/tfds_covid19/covid19_dataset/incomplete.RUUBRZ_1.0.0/covid19_dataset-train.tfrecord*...:   …

Generating eval examples...: 0 examples [00:00, ? examples/s]

Shuffling /content/tfds_covid19/covid19_dataset/incomplete.RUUBRZ_1.0.0/covid19_dataset-eval.tfrecord*...:   0…

Dataset covid19_dataset downloaded and prepared to /content/tfds_covid19/covid19_dataset/1.0.0. Subsequent calls will reuse this data.

Dataset built successfully.
tfds.core.DatasetInfo(
    name='covid19_dataset',
    full_name='covid19_dataset/1.0.0',
    description="""
    COVID-19 global time-series dataset built from Johns Hopkins University
    CSSE data. Contains daily confirmed cases, deaths, recovered, new cases,
    new deaths, and 7-day rolling averages per country from 2020-01-22 to
    2023-03-09. Cleaned and aggregated to country level.
    """,
    homepage='https://www.tensorflow.org/datasets/catalog/covid19_dataset',
    data_dir='/content/tfds_covid19/covid19_dataset/1.0.0',
    file_format=tfrecord,
    download_size=Unknown size,
    dataset_size=39.44 MiB,
    features=FeaturesDict({
        'confirmed': int64,
        'country': Text(shape=(), dtype=string),
        'date': Text(shape=(), dtype=string),
        'deaths': int64,
        'new_cases': int64,
       

In [17]:
# Load and validate both splits
ds_train = builder.as_dataset(split='train', shuffle_files=False)
ds_eval  = builder.as_dataset(split='eval',  shuffle_files=False)

# Count records in each split
train_count = ds_train.cardinality().numpy()
eval_count  = ds_eval.cardinality().numpy()

print(f"Train split  : {train_count:,} records")
print(f"Eval split   : {eval_count:,} records")
print(f"Total        : {train_count + eval_count:,} records")
print(f"Train %      : {train_count / (train_count + eval_count) * 100:.1f}%")

# Peek at one record from each split
print("\nSample train record:")
for record in ds_train.take(1):
    for k, v in record.items():
        print(f"  {k:15s}: {v.numpy()}")

print("\nSample eval record:")
for record in ds_eval.take(1):
    for k, v in record.items():
        print(f"  {k:15s}: {v.numpy()}")

Train split  : 183,794 records
Eval split   : 45,949 records
Total        : 229,743 records
Train %      : 80.0%

Sample train record:
  confirmed      : 13
  country        : b'Botswana'
  date           : b'2020-04-12'
  deaths         : 1
  new_cases      : 0
  new_deaths     : 0
  recovered      : 0
  rolling_7day   : 1.0

Sample eval record:
  confirmed      : 15584
  country        : b'Marshall Islands'
  date           : b'2023-02-09'
  deaths         : 17
  new_cases      : 0
  new_deaths     : 0
  recovered      : 0
  rolling_7day   : 0.0


## TensorFlow Dataset Summary

- Format      : TFRecord (tfds.core.GeneratorBasedBuilder)
- Version     : 1.0.0
- Dataset size: 39.44 MiB
- Total records: 229,743

| Split | Records | Date Range |
|---|---|---|
| Train (80%) | 183,794 | 2020-01-22 → 2022-07-24 |
| Eval  (20%) |  45,949 | 2022-07-24 → 2023-03-09 |

Split is chronological (on purpose) to preserve time-series integrity.
Features: country, date, confirmed, deaths, recovered,
          new_cases, new_deaths, rolling_7day

Statistical Analysis

In [18]:
# Extract training split for analysis
train_df = master[master['date'] <= '2022-07-24'].copy()
eval_df  = master[master['date'] >  '2022-07-24'].copy()

print(f"Train rows : {len(train_df):,}")
print(f"Eval rows  : {len(eval_df):,}")
print(f"Train date range: {train_df['date'].min().date()} → {train_df['date'].max().date()}")
print(f"Eval date range : {eval_df['date'].min().date()} → {eval_df['date'].max().date()}")

analysis_cols = ['confirmed', 'deaths', 'recovered', 'new_cases', 'new_deaths', 'rolling_7day']
print(f"\nAnalysis columns: {analysis_cols}")

Train rows : 183,915
Eval rows  : 45,828
Train date range: 2020-01-22 → 2022-07-24
Eval date range : 2022-07-25 → 2023-03-09

Analysis columns: ['confirmed', 'deaths', 'recovered', 'new_cases', 'new_deaths', 'rolling_7day']


In [20]:
# Mean, Standard Deviation, Variance
print("=" * 60)
print("STATISTICS — TRAINING SPLIT")
print("=" * 60)

stats = {}
for col in analysis_cols:
    mean = train_df[col].mean()
    std  = train_df[col].std()
    var  = train_df[col].var()
    stats[col] = {'mean': mean, 'std': std, 'variance': var}
    print(f"\n{col.upper()}")
    print(f"  Mean     : {mean:>15,.2f}")
    print(f"  Std Dev  : {std:>15,.2f}")
    print(f"  Variance : {var:>15,.2f}")

STATISTICS — TRAINING SPLIT

CONFIRMED
  Mean     :      934,621.32
  Std Dev  :    4,326,638.97
  Variance : 18,719,804,782,686.80

DEATHS
  Mean     :       15,802.94
  Std Dev  :       65,318.90
  Variance : 4,266,559,227.96

RECOVERED
  Mean     :      127,729.96
  Std Dev  :      914,919.50
  Variance : 837,077,696,421.74

NEW_CASES
  Mean     :        3,105.42
  Std Dev  :       18,809.94
  Variance :  353,813,796.44

NEW_DEATHS
  Mean     :           34.86
  Std Dev  :          174.51
  Variance :       30,455.47

ROLLING_7DAY
  Mean     :        3,090.60
  Std Dev  :       17,340.20
  Variance :  300,682,564.22


In [21]:
# ADD (Average Absolute Deviation) and MAD (Median Absolute Deviation)
print("=" * 60)
print("ADD & MAD — TRAINING SPLIT")
print("=" * 60)

for col in analysis_cols:
    values = train_df[col]

    ADD = np.mean(np.abs(values - values.mean()))
    MAD = np.median(np.abs(values - values.median()))

    stats[col]['ADD'] = ADD
    stats[col]['MAD'] = MAD

    print(f"\n{col.upper()}")
    print(f"  ADD (Avg Absolute Deviation)    : {ADD:>15,.2f}")
    print(f"  MAD (Median Absolute Deviation) : {MAD:>15,.2f}")

ADD & MAD — TRAINING SPLIT

CONFIRMED
  ADD (Avg Absolute Deviation)    :    1,414,919.30
  MAD (Median Absolute Deviation) :       30,715.00

DEATHS
  ADD (Avg Absolute Deviation)    :       24,005.98
  MAD (Median Absolute Deviation) :          472.00

RECOVERED
  ADD (Avg Absolute Deviation)    :      212,565.97
  MAD (Median Absolute Deviation) :            2.00

NEW_CASES
  ADD (Avg Absolute Deviation)    :        4,947.34
  MAD (Median Absolute Deviation) :           44.00

NEW_DEATHS
  ADD (Avg Absolute Deviation)    :           55.12
  MAD (Median Absolute Deviation) :            0.00

ROLLING_7DAY
  ADD (Avg Absolute Deviation)    :        4,847.63
  MAD (Median Absolute Deviation) :           72.14


In [22]:
# Full statistics summary table
print("=" * 60)
print("FULL STATISTICS SUMMARY TABLE")
print("=" * 60)

summary = pd.DataFrame(stats).T
summary.columns = ['Mean', 'Std Dev', 'Variance', 'ADD', 'MAD']
summary = summary.round(2)

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print(summary.to_string())

FULL STATISTICS SUMMARY TABLE
                   Mean      Std Dev              Variance          ADD       MAD
confirmed    934,621.32 4,326,638.97 18,719,804,782,686.80 1,414,919.30 30,715.00
deaths        15,802.94    65,318.90      4,266,559,227.96    24,005.98    472.00
recovered    127,729.96   914,919.50    837,077,696,421.74   212,565.97      2.00
new_cases      3,105.42    18,809.94        353,813,796.44     4,947.34     44.00
new_deaths        34.86       174.51             30,455.47        55.12      0.00
rolling_7day   3,090.60    17,340.20        300,682,564.22     4,847.63     72.14


In [23]:
# Distribution analysis (skewness and kurtosis)
from scipy import stats as scipy_stats

print("=" * 60)
print("DISTRIBUTION ANALYSIS — TRAINING SPLIT")
print("=" * 60)

for col in analysis_cols:
    values = train_df[col]
    skewness = scipy_stats.skew(values)
    kurtosis = scipy_stats.kurtosis(values)
    median   = values.median()
    q25      = values.quantile(0.25)
    q75      = values.quantile(0.75)
    iqr      = q75 - q25

    print(f"\n{col.upper()}")
    print(f"  Median   : {median:>15,.2f}")
    print(f"  Q25      : {q25:>15,.2f}")
    print(f"  Q75      : {q75:>15,.2f}")
    print(f"  IQR      : {iqr:>15,.2f}")
    print(f"  Skewness : {skewness:>15,.4f}  {'(right-skewed)' if skewness > 0 else '(left-skewed)'}")
    print(f"  Kurtosis : {kurtosis:>15,.4f}  {'(heavy tails)' if kurtosis > 0 else '(light tails)'}")

DISTRIBUTION ANALYSIS — TRAINING SPLIT

CONFIRMED
  Median   :       30,715.00
  Q25      :        1,591.00
  Q75      :      316,424.50
  IQR      :      314,833.50
  Skewness :         10.9308  (right-skewed)
  Kurtosis :        158.8615  (heavy tails)

DEATHS
  Median   :          472.00
  Q25      :           21.00
  Q75      :        5,325.00
  IQR      :        5,304.00
  Skewness :          8.5045  (right-skewed)
  Kurtosis :         90.3873  (heavy tails)

RECOVERED
  Median   :            2.00
  Q25      :            0.00
  Q75      :       11,098.00
  IQR      :       11,098.00
  Skewness :         19.3501  (right-skewed)
  Kurtosis :        497.0975  (heavy tails)

NEW_CASES
  Median   :           44.00
  Q25      :            0.00
  Q75      :          695.00
  IQR      :          695.00
  Skewness :         20.7661  (right-skewed)
  Kurtosis :        742.4141  (heavy tails)

NEW_DEATHS
  Median   :            0.00
  Q25      :            0.00
  Q75      :            8.00
 

Visualization